In [2]:
from dotenv import load_dotenv
import os
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, List
from langchain_google_genai import ChatGoogleGenerativeAI
from pydantic import BaseModel, Field
from langchain.agents import create_agent

In [3]:
load_dotenv()
api_key = os.getenv("google_api_key")

# **Models**

In [4]:
scene_creator = ChatGoogleGenerativeAI(model="gemini-2.5-flash", api_key=api_key, temperature=0.7)
evalvator = ChatGoogleGenerativeAI(model="gemini-2.5-flash", api_key=api_key, temperature=0.7)
image_prompt_creator = ChatGoogleGenerativeAI(model="gemini-2.5-flash", api_key=api_key, temperature=0.7)
eval_image_prompt = ChatGoogleGenerativeAI(model="gemini-2.5-flash", api_key=api_key, temperature=0.7)

# **Structured output**

In [5]:
class baseScene(BaseModel):
    scene_number: int = Field(description="A simple numeric label used to keep the sequence organized.")
    location: str = Field(description="A short note describing where the scene takes place — could be indoor, outdoor, real place, fictional place, or a general setting.Example: “Inside a small office,” “Busy marketplace,” “Temple courtyard,” “Rural farmland,” etc.")
    time_of_the_day: str = Field(description="Specifies the lighting and mood. Example: “Early morning,” “Sunset,” “Nighttime,” “Midday,” etc.")
    characters_present: str = Field(description="A list of all people (or entities) appearing in the scene. Example: “Main character only,” “Thiruvalluvar,” “Two farmers,” “Student and teacher,” etc.")
    brief_description: str = Field(description="A 1–2 sentence overview of what’s visually happening. Example: “The character walks into the room, observing the surroundings,” or “A peaceful sunrise washes over the village.”")
    dialogue_summary:str = Field(description="A quick line describing what is spoken or conveyed. Example: “The character explains the core idea,” or “Thiruvalluvar delivers a short line about wisdom.”")

class listScenes(BaseModel):
    scenes:List[baseScene] = Field(description="A collection of scene objects, each containing structured details such as scene number, location, time of day, characters, brief description, and dialogue summary. This list represents the full storyboard or sequence of scenes for the video or narrative.")

class evalReportScenes(BaseModel):
    is_approved_scenes: bool = Field(description="Boolean value used to define whether the scenes are approved or not")
    correction: str = Field(description="Contains the feedback to improve scenes.")

class evalReportImagePrompt(BaseModel):
    is_approved_image_prompt: bool = Field(description="Boolean value used to define whether the scenes are approved or not")
    correction: str = Field(description="Contains the feedback to improve scenes.")

class characterConsistenyModel(BaseModel):
    identity_tags: List[str] = Field(description="Short factual phrases that describe persistent character attributes (face marks, hair style, clothing details). Use concise, repeatable tags.")
    style_lock: str = Field(description="Short instruction to lock or strongly prefer the same overall art style across outputs (e.g., painterly warm-tone).")

class imagePromptModel(BaseModel):
    image_number: int = Field(description="A simple numeric label used to keep the sequence organized.")
    title: str = Field(description="Short human-readable title for quick identification in UIs and logs. Keep it concise (<= 60 chars).")
    main_subject: str = Field(description="Clear, factual one-line description of the primary subject (who/what). Focus on identity and action rather than style.")
    visual_style: str = Field(description="High-level art direction: medium, camera lens, lighting, mood, and stylistic adjectives. Keeps the overall look consistent across generations.")
    details: List[str] = Field(description="List of short, specific attributes to enforce appearance and environment (appearance, clothing, props, background constraints). Each item should be a concise noun-phrase.")
    negative_prompts: List[str] = Field(description="List of terms/concepts to avoid (artifacts, unwanted objects, quality issues). Use specific items such as 'text', 'watermark', 'extra limbs', 'deformed hands'.")
    character_consistency: characterConsistenyModel = Field(description="hints to preserve the same character across multiple generations")

class imagePromptList(BaseModel):
    image_prompt: List[imagePromptModel] = Field(description="List of all image prompts for the scenes")

# **Agents**

In [6]:
scene_agent = create_agent(
    model=scene_creator,
    response_format=listScenes
)

eval_agent = create_agent(
    model=evalvator,
    response_format=evalReportScenes
)

image_prompt_agent = create_agent(
    model=image_prompt_creator,
    response_format=imagePromptList
)

image_eval_agent = create_agent(
    model=eval_image_prompt,
    response_format=evalReportImagePrompt
)

# **1. State**
- Shared data structure that flows through the workflow

In [7]:
class AgentState(TypedDict):
    script: str
    scenes: List[dict]
    image_prompt: list[dict]
    eval_result_scenes: str
    eval_result_image_prompt: str
    is_approved_scenes: bool
    is_approved_image_prompt: bool
    revision_count_scenes: int
    revision_count_image_prompt: int

# **Graph**

In [30]:
workflow = StateGraph(AgentState)

# **2. Nodes**
- Functions that perform work and update the state
- Each node get state as input and return updated state as output

In [31]:
def scene_creator_agent(state: AgentState):
    """Agent 1: That converts script to scenes"""

    script = state["script"]

    prompt = f"""You are a script breakdown specialist. Convert the following script into scenes and give json prompt.
    
        For each scene, provide:
        - Scene number
        - Location
        - Time of day
        - Characters present
        - Brief description
        - Dialogue summary

        Script:
        {script}

        Format as a structured list of json prompts and give 30 scenes."""
    
    response = scene_agent.invoke({
        "messages": [{"role": "user", "content": prompt}]
    })

    list_of_scenes = [{
        "scene_number": i.scene_number,
        "location": i.location,
        "time_of_day": i.time_of_the_day,
        "characters_present": i.characters_present,
        "brief_description": i.brief_description,
        "dialogue_summary": i.dialogue_summary
    } for i in response["structured_response"].scenes]

    return {
        "scenes": list_of_scenes,
        "revision_count_scenes": state.get("revision_count_scenes") + 1
    }

def eval_scene_agent(state: AgentState):
    """Agent 2: That evalvates generated scenes and provide feedback if not okay""" 
    scenes = state["scenes"] 
    rules = """
    Rules to check:
    1. Each scene must have a clear location
    2. Each scene must specify time of day
    3. Characters must be clearly identified
    4. Scene transitions must be logical
    5. Each scene should have a clear purpose
    """
    prompt = f"""You are a script supervisor. Evaluate these scenes against the rules.
    
    {rules}

    Scenes to evaluate:
    {scenes}

    Provide:
    1. PASS or FAIL for each rule
    2. Specific issues found
    3. Overall recommendation: APPROVED or NEEDS_REVISION
    4. Suggestions for improvement if needed"""  

    response = eval_agent.invoke({
    "messages": [{"role": "user", "content": prompt}]
    })

    return {
        "is_approved_scenes": response["structured_response"].is_approved_scenes,
        "eval_result_scenes": response["structured_response"].correction
    }

def regenerate_agent(state: AgentState):
    """Agent 3: Regenerate the scenes based on given feedback"""
    scenes = state["scenes"]
    correction = state["eval_result_scenes"]

    prompt = f"""Revise these scenes based on the evaluation feedback:

    Original Scenes:
    {scenes}

    Evaluation Feedback:
    {correction}

    Provide improved scenes that address all issues."""

    response = scene_agent.invoke({
    "messages": [{"role": "user", "content": prompt}]
    })

    list_of_scenes = [{
        "scene_number": i.scene_number,
        "location": i.location,
        "time_of_day": i.time_of_the_day,
        "characters_present": i.characters_present,
        "brief_description": i.brief_description,
        "dialogue_summary": i.dialogue_summary
    } for i in response["structured_response"].scenes]

    return {
        "scenes": list_of_scenes,
        "revision_count_scenes": state.get("revision_count_scenes") + 1
    }

def should_loop(state: AgentState):
    """Decides if revisions need or not"""
    if state["is_approved_scenes"]:
        return "end"
    elif state["revision_count_scenes"] > 5:
        return "end"
    else:
        return "revise"
    
def generate_image_prompt_agent(state: AgentState):
    """Generates scenes into image generation json prompt"""
    scenes = state["scenes"]
    prompt = f"""You are an expert at creating structured image generation prompts for AI image models.

                Given the following scene breakdown, create a detailed image prompt for EACH scene following this exact structure:

                Scenes:
                {scenes}

                For each scene, you must create an ImagePromptModel with these fields:

                1. **image_number**: Sequential number (1, 2, 3...)

                2. **title**: Short descriptive title (<= 60 chars)
                - Example: "Sarah Working Alone at Coffee Shop"

                3. **main_subject**: One clear sentence describing who/what and their action
                - Focus on IDENTITY and ACTION, not style
                - Example: "Young woman typing intensely on laptop at corner table"

                4. **visual_style**: Art direction combining medium, camera, lighting, mood
                - Include: medium type (photo-realistic, cinematic, etc.)
                - Camera details: lens type, angle, shot composition
                - Lighting: quality, direction, time of day
                - Mood: emotional tone, color palette
                - Example: "Cinematic photo-realistic, 35mm film, medium shot from slight overhead angle, soft morning sunlight streaming through windows, warm golden hour lighting, amber and cream color palette, shallow depth of field, moody and tense atmosphere"

                5. **details**: List of specific visual attributes (5-10 items)
                - Character appearance details
                - Clothing descriptions
                - Props and objects
                - Environment/background specifics
                - Spatial relationships
                - Example: ["casual business attire", "laptop with glowing screen", "wooden table surface", "coffee cup nearby", "large windows in background", "urban coffee shop interior", "morning sunlight rays", "blurred background with cafe customers"]

                6. **negative_prompts**: List of things to avoid (5-10 items)
                - Common AI artifacts
                - Quality issues
                - Unwanted elements
                - Example: ["text", "watermark", "blurry", "low quality", "deformed hands", "extra limbs", "distorted face", "unrealistic proportions", "oversaturated", "anime style"]

                7. **character_consistency**: Object with two sub-fields:
                
                a. **identity_tags**: List of persistent character attributes (3-7 tags)
                    - Physical features that stay consistent
                    - Clothing style markers
                    - Distinctive characteristics
                    - Example: ["shoulder-length brown hair", "green eyes", "slim build", "navy blazer", "silver necklace", "focused expression"]
                
                b. **style_lock**: One instruction for consistent art style
                    - Locks the overall visual treatment
                    - Example: "cinematic photo-realistic with warm natural lighting and film grain"

                IMPORTANT GUIDELINES:
                - Keep identity_tags factual and repeatable across scenes
                - Make main_subject action-focused, not style-focused
                - visual_style should be consistent across all scenes for the same project
                - details should be specific noun-phrases, not full sentences
                - negative_prompts should target common AI generation issues
                - character_consistency helps maintain same character appearance across multiple images

                Return the data as a JSON array of image prompt objects."""
    
    response = image_prompt_agent.invoke({
    "messages": [{"role": "user", "content": prompt}]
    })

    result = [{
    "image_number": i.image_number,
    "title": i.title,
    "main_subject": i.main_subject,
    "visual_style": i.visual_style,
    "details": i.details,
    "negative_prompts": i.negative_prompts,
    "character_consistency": {"identity_tags": i.character_consistency.identity_tags, "style_lock": i.character_consistency.style_lock}
    } for i in response["structured_response"].image_prompt]

    return {"image_prompt": result,
            "revision_count_image_prompt": state["revision_count_image_prompt"] + 1}

def eval_image_prompt_agent(state: AgentState):
    """Evalvate structured image prompts for quality and accuracy"""
    scenes = state["scenes"]
    image_propt = state["image_prompt"]

    prompt = f"""You are evaluating structured image generation prompts against original scenes.

    Original Scenes:
    {scenes}

    Generated image prompt:
    {image_propt}

    Evaluate EACH image prompt based on these criteria:

    **STRUCTURE VALIDATION:**
    1. ✓ All required fields present? (image_number, title, main_subject, visual_style, details, negative_prompts, character_consistency)
    2. ✓ Title <= 60 characters?
    3. ✓ details list has 5-10 items?
    4. ✓ negative_prompts list has 5-10 items?
    5. ✓ identity_tags has 3-7 items?
    6. ✓ style_lock is a single clear instruction?

    **CONTENT ACCURACY:**
    7. ✓ image_number sequential and correct?
    8. ✓ main_subject matches scene action and characters?
    9. ✓ visual_style includes time of day from scene?
    10. ✓ details include all key characters mentioned in scene?
    11. ✓ details include location-specific elements (INT/EXT)?

    **QUALITY CHECKS:**
    12. ✓ main_subject is factual and action-focused (not style-focused)?
    13. ✓ visual_style is comprehensive (medium, camera, lighting, mood)?
    14. ✓ details are concise noun-phrases?
    15. ✓ identity_tags are consistent and repeatable?
    16. ✓ negative_prompts target common AI issues?
    17. ✓ character_consistency preserves character across scenes?

    **CONSISTENCY CHECKS:**
    18. ✓ visual_style consistent across all scenes?
    19. ✓ style_lock same for all scenes in project?
    20. ✓ identity_tags for same character consistent across scenes?
    
    Provide:
    1. PASS or FAIL for each rule
    2. Specific issues found
    3. Overall recommendation: APPROVED or NEEDS_REVISION
    4. Suggestions for improvement if needed"""

    response = image_eval_agent.invoke({
    "messages": [{"role": "user", "content": prompt}]
    })

    return {
        "is_approved_image_prompt": response["structured_response"].is_approved_image_prompt,
        "eval_result_image_prompt": response["structured_response"].correction
    }

def image_prompt_regenerate_agent(state: AgentState):
    """Regenerate image prompt based on given correction"""
    image_prompt = state["image_prompt"]
    scenes = state["scenes"]
    corrections = state["eval_result_image_prompt"]

    prompt = f"""Revise these image prompt based on the evaluation feedback:

    Original Scenes:
    {scenes}

    generated image prompt:
    {image_prompt}

    evaluation feedback:
    {corrections}

    Provide improved image prompt that address all issues.""" 

    response = image_prompt_agent.invoke({
    "messages": [{"role": "user", "content": prompt}]
    })

    result = [{
    "image_number": i.image_number,
    "title": i.title,
    "main_subject": i.main_subject,
    "visual_style": i.visual_style,
    "details": i.details,
    "negative_prompts": i.negative_prompts,
    "character_consistency": {"identity_tags": i.character_consistency.identity_tags, "style_lock": i.character_consistency.style_lock}
    } for i in response["structured_response"].image_prompt]

    return {
        "image_prompt": result,
        "revision_count_image_prompt": state["revision_count_image_prompt"] + 1
    }

def decision_for_image_prompt(state: AgentState):
    """Decides wether the loop continues or not"""
    if state["is_approved_image_prompt"]:
        return "end"
    
    elif state["revision_count_image_prompt"] > 5:
        return "end"
    
    else:
        return "revise"

In [33]:
workflow.add_node("create_scene", scene_creator_agent)
workflow.add_node("eval_scene", eval_scene_agent)
workflow.add_node("regenerate_scene", regenerate_agent)
workflow.add_node("generate_image_prompt_agent", generate_image_prompt_agent)
workflow.add_node("eval_image_prompt_agent", eval_image_prompt_agent)
workflow.add_node("image_prompt_regenerate_agent", image_prompt_regenerate_agent)

ValueError: Node `create_scene` already present.

# **Edges**
- Connection between nodes
- -> Normal edges: direct connection
- -> Conditional edges: dynamic routing based on state

In [34]:
workflow.add_edge(START, "create_scene")
workflow.add_edge("create_scene", "eval_scene")
workflow.add_conditional_edges("eval_scene", should_loop, {"end": "generate_image_prompt_agent", "revise": "regenerate_scene"})
workflow.add_edge("regenerate_scene", "eval_scene")

workflow.add_edge("generate_image_prompt_agent", "eval_image_prompt_agent")
workflow.add_conditional_edges("eval_image_prompt_agent", decision_for_image_prompt, {"end": END, "revise": "image_prompt_regenerate_agent"})
workflow.add_edge("image_prompt_regenerate_agent", "eval_image_prompt_agent")

In [35]:
app = workflow.compile()

# **Check**

With DEBUG

In [54]:
final_result = {}
final_state = {}

for i in app.stream(
    {
        "script": "Ever spent hours studying and still remembered nothing the next day? So today, I’m giving you five study tips that actually work, backed by science. First, use the 25–5 rule: study for 25 minutes with zero distractions, then rest for 5 minutes — it keeps your brain fresh. Next, teach what you learned to someone else; if you can explain it simply, you truly understand it, and even talking to a wall works. Use active recall by closing your book and trying to remember the key ideas instead of rereading everything. Keep your study sessions short and consistent because short bursts help you absorb more than long, exhausting sessions. And finally, keep your notes simple with keywords, bullet points, and quick diagrams — simple notes make revision faster. In the end, studying smart always beats studying hard, so keep learning and stay consistent.",
        "scenes": [],
        "eval_result_scenes": None,
        "is_approved_scenes": None,
        "revision_count_scenes": 0,
        "image_prompt": [],
        "eval_result_image_prompt": None,
        "is_approved_image_prompt": None,
        "revision_count_image_prompt": 0
    },
    stream_mode=["debug", "updates", "values"]
):
    if i[0] == "debug" and i[1]['type'] == "task":
        print(f"{i[0]} - step: {i[1]['step']}, type: {i[1]['type']}, name: {i[1]['payload']['name']}")
        first_name = i[1]['payload']['name']

    if i[0] == "debug" and i[1]['type'] == "task_result":
        print(f"{i[0]} - step: {i[1]['step']}, type: {i[1]['type']}, name: {i[1]['payload']['name']}, error: {i[1]['payload']['error']}, result: {i[1]['payload']['result']}")
        first_name = i[1]['payload']['name']

    if i[0] == "updates":
        keys = list(i[1][first_name].keys())
        print(f"{i[0]}, keys: {list(i[1][first_name].keys())}, dict: {i[1][first_name]}")
        for j in keys:
            final_result[j] = i[1][first_name][j]
        
    if i[0] == "values":
        final_state = i[1]

    

debug - step: 1, type: task, name: create_scene
updates, keys: ['scenes', 'revision_count_scenes'], dict: {'scenes': [{'scene_number': 1, 'location': 'College library, late evening', 'time_of_day': 'Nighttime', 'characters_present': 'A frustrated student', 'brief_description': 'A student stares blankly at a pile of books, head in hands, clearly overwhelmed and exhausted.', 'dialogue_summary': "Narrator asks, 'Ever spent hours studying and still remembered nothing the next day?'"}, {'scene_number': 2, 'location': 'Bright, modern study room', 'time_of_day': 'Morning', 'characters_present': 'Narrator (off-screen voice), a hopeful student', 'brief_description': 'A new day dawns, sunlight streams into a clean study space, hinting at a fresh start.', 'dialogue_summary': "Narrator states, 'Today, I’m giving you five study tips that actually work, backed by science.'"}, {'scene_number': 3, 'location': 'Animated infographic background', 'time_of_day': 'Daytime', 'characters_present': 'None (foc

In [57]:
len(final_state.keys())

9

In [52]:
print(len(list(final_result.keys())))

found = list(final_result.keys())
actual = ["script", "scenes", "image_prompt", "eval_result_scenes", "eval_result_image_prompt", "is_approved_scenes", "is_approved_image_prompt", "revision_count_scenes", "revision_count_image_prompt"]

for i in actual:
    if i not in found:
        print(i)

8
script


Without DEBUG

In [ ]:
result = app.invoke(
    {
        "script": "Ever spent hours studying and still remembered nothing the next day? So today, I’m giving you five study tips that actually work, backed by science. First, use the 25–5 rule: study for 25 minutes with zero distractions, then rest for 5 minutes — it keeps your brain fresh. Next, teach what you learned to someone else; if you can explain it simply, you truly understand it, and even talking to a wall works. Use active recall by closing your book and trying to remember the key ideas instead of rereading everything. Keep your study sessions short and consistent because short bursts help you absorb more than long, exhausting sessions. And finally, keep your notes simple with keywords, bullet points, and quick diagrams — simple notes make revision faster. In the end, studying smart always beats studying hard, so keep learning and stay consistent.",
        "scenes": [],
        "eval_result_scenes": None,
        "is_approved_scenes": None,
        "revision_count_scenes": 0,
        "image_prompt": [],
        "eval_result_image_prompt": None,
        "is_approved_image_prompt": None,
        "revision_count_image_prompt": 0
    }
)

In [24]:
scemes = result["scenes"]

# **Seprate testing**

In [26]:
prompt = f"""You are an expert at creating structured image generation prompts for AI image models.

            Given the following scene breakdown, create a detailed image prompt for EACH scene following this exact structure:

                Scenes:
                {scemes}

                Generate 30 image prompt because there is 30 scenes(1 image prompt for 1 scene)

                For each scene, you must create an ImagePromptModel with these fields:

                1. **image_number**: Sequential number (1, 2, 3...)

                2. **title**: Short descriptive title (<= 60 chars)
                - Example: "Sarah Working Alone at Coffee Shop"

                3. **main_subject**: One clear sentence describing who/what and their action
                - Focus on IDENTITY and ACTION, not style
                - Example: "Young woman typing intensely on laptop at corner table"

                4. **visual_style**: Art direction combining medium, camera, lighting, mood
                - Include: medium type (photo-realistic, cinematic, etc.)
                - Camera details: lens type, angle, shot composition
                - Lighting: quality, direction, time of day
                - Mood: emotional tone, color palette
                - Example: "Cinematic photo-realistic, 35mm film, medium shot from slight overhead angle, soft morning sunlight streaming through windows, warm golden hour lighting, amber and cream color palette, shallow depth of field, moody and tense atmosphere"

                5. **details**: List of specific visual attributes (5-10 items)
                - Character appearance details
                - Clothing descriptions
                - Props and objects
                - Environment/background specifics
                - Spatial relationships
                - Example: ["casual business attire", "laptop with glowing screen", "wooden table surface", "coffee cup nearby", "large windows in background", "urban coffee shop interior", "morning sunlight rays", "blurred background with cafe customers"]

                6. **negative_prompts**: List of things to avoid (5-10 items)
                - Common AI artifacts
                - Quality issues
                - Unwanted elements
                - Example: ["text", "watermark", "blurry", "low quality", "deformed hands", "extra limbs", "distorted face", "unrealistic proportions", "oversaturated", "anime style"]

                7. **character_consistency**: Object with two sub-fields:
                
                a. **identity_tags**: List of persistent character attributes (3-7 tags)
                    - Physical features that stay consistent
                    - Clothing style markers
                    - Distinctive characteristics
                    - Example: ["shoulder-length brown hair", "green eyes", "slim build", "navy blazer", "silver necklace", "focused expression"]
                
                b. **style_lock**: One instruction for consistent art style
                    - Locks the overall visual treatment
                    - Example: "cinematic photo-realistic with warm natural lighting and film grain"

                IMPORTANT GUIDELINES:
                - Keep identity_tags factual and repeatable across scenes
                - Make main_subject action-focused, not style-focused
                - visual_style should be consistent across all scenes for the same project
                - details should be specific noun-phrases, not full sentences
                - negative_prompts should target common AI generation issues
                - character_consistency helps maintain same character appearance across multiple images

                Return the data as a JSON array of image prompt objects."""

model_here = image_prompt_creator.with_structured_output(imagePromptList)

response = model_here.invoke(prompt)

In [28]:
res = image_prompt_agent.invoke({
        "messages": [{"role": "user", "content": prompt}]
    })

In [32]:
res

{'messages': [HumanMessage(content='You are an expert at creating structured image generation prompts for AI image models.\n\n            Given the following scene breakdown, create a detailed image prompt for EACH scene following this exact structure:\n\n                Scenes:\n                [{\'scene_number\': 1, \'location\': \'College dorm room\', \'time_of_day\': \'Late night\', \'characters_present\': \'Frustrated Student\', \'brief_description\': \'A student is hunched over a desk, surrounded by open books and notes, looking exhausted and rubbing their temples in frustration.\', \'dialogue_summary\': "Voiceover: \'Ever spent hours studying and still remembered nothing the next day?\'"}, {\'scene_number\': 2, \'location\': "Student\'s bedroom", \'time_of_day\': \'Early morning\', \'characters_present\': \'Frustrated Student\', \'brief_description\': "The same student wakes up, looks blankly at the ceiling, trying to recall yesterday\'s study material but drawing a blank.", \'d

In [35]:
result = [{
    "image_number": i.image_number,
    "title": i.title,
    "main_subject": i.main_subject,
    "visual_style": i.visual_style,
    "details": i.details,
    "negative_prompts": i.negative_prompts,
    "character_consistency": {"identity_tags": i.character_consistency.identity_tags, "style_lock": i.character_consistency.style_lock}
} for i in res["structured_response"].image_prompt]

In [36]:
result

[{'image_number': 1,
  'title': 'Frustrated Student Studying Late Night',
  'main_subject': 'Young adult male student hunched over a desk, rubbing temples in frustration, surrounded by open books and notes.',
  'visual_style': 'Cinematic photo-realistic, 50mm lens, medium shot, dim desk lamp lighting, dark shadows, tense and exhausted mood, muted color palette.',
  'details': ['college dorm room interior',
   'wooden desk',
   'open textbooks',
   'scattered notes',
   'tired eyes',
   'messy hair',
   'casual t-shirt',
   'empty coffee mug',
   'late night atmosphere',
   'cluttered background'],
  'negative_prompts': ['text',
   'watermark',
   'blurry',
   'low quality',
   'deformed hands',
   'extra limbs',
   'distorted face',
   'unrealistic proportions',
   'oversaturated',
   'anime style',
   'cartoon',
   'illustration',
   'bad anatomy',
   'poorly drawn',
   'mutations',
   'disfigured'],
  'character_consistency': {'identity_tags': ['young adult male',
    'short dark hai